<a href="https://colab.research.google.com/github/mlbyhoang/marketing-attribution/blob/main/notebooks/02_exploratory_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 – Exploratory Data Analysis

In [ ]:
# Load libraries
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Connect to Google Drive
from google.colab import drive, userdata
drive.mount('/content/drive')

In [ ]:
# Load processed data path from user data / secrets
PROCESSED_DATA_PATH = userdata.get("MTA_PROCESSED_PATH")

In [ ]:
# Load processed dataset and inspect structure
df5 = pd.read_parquet(PROCESSED_DATA_PATH)

print("Shape:", df5.shape)
print("Columns:", df5.columns.tolist())

In [ ]:
# Display first few rows
df5.head()

In [ ]:
# Display DataFrame summary and data types
df5.info()

In [ ]:
# Display summary statistics
df5.describe()

In [ ]:
# Unnest sequences and get channel counts
channels = df5['sequence'].explode()

channels.value_counts()

In [ ]:
# Calculate channel distribution in percentage
channel_distribution = (
    channels
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

channel_distribution

In [ ]:
# Calculate first and last touch percentages
print("First Touch:")
print(
    df5["sequence"]
    .str[0]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nLast Touch:")
print(
    df5["sequence"]
    .str[-1]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
# Distribution of total touchpoints
df5["touchpoint_sum"].plot(
    kind="box",
    title="Touchpoints Verteilung",
    figsize=(10, 10)
)
plt.ylabel("Anzahl Touchpoints")
plt.show()

In [ ]:
# Check for outliers
print(df5['touchpoint_sum'].quantile([0.97, 0.98, 0.99, 1.0]))
print(f"\nMax: {df5['touchpoint_sum'].max()}")
print(f"Sequenzen > 10:  {(df5['touchpoint_sum'] > 10).sum()}")
print(f"Sequenzen > 20:  {(df5['touchpoint_sum'] > 20).sum()}")
print(f"Sequenzen > 50:  {(df5['touchpoint_sum'] > 50).sum()}")

In [ ]:
# Remove outliers
df6 = df5[df5['touchpoint_sum'] <= 10].copy()

print(f"Vorher:  {len(df5)} Sequenzen")
print(f"Nachher: {len(df6)} Sequenzen")
print(f"Entfernt: {len(df5) - len(df6)} Sequenzen ({(1 - len(df6)/len(df5)):.2%})")
print(f"\nConversion Rate vorher:  {df5['order'].mean():.2%}")
print(f"Conversion Rate nachher: {df6['order'].mean():.2%}")
print(f"\nMax Sequenzlänge: {df6['touchpoint_sum'].max()}")

In [ ]:
# Prepare data
seq_lengths = df6['touchpoint_sum'].values
converted   = df6['order'].values

max_length = seq_lengths.max()
lengths = np.arange(1, max_length + 1)

# Count of total and converted sequences by length
n_sequences = np.array([(seq_lengths == l).sum() for l in lengths])
n_converted = np.array([((seq_lengths == l) & (converted == 1)).sum() for l in lengths])

# # Conversion rate by length (non-empty sequences only)
with np.errstate(divide='ignore', invalid='ignore'):
    conversion_rate = np.where(n_sequences > 0, n_converted / n_sequences, np.nan)

# Chart 1: Number of sequences by length (log scale)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.plot(lengths, np.log1p(n_sequences), color='purple', label='Anzahl Sequenzen')
ax1.plot(lengths, np.log1p(n_converted), color='red', label='Anzahl konvertierter Sequenzen')
ax1.set_xlabel('Länge der Customer Journey')
ax1.set_ylabel('Logarithmierte Anzahl (konvertierter) Sequenzen')
ax1.set_title('Anzahl der Sequenzen nach Länge')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Chart 2: Conversion rate by sequence length
ax2 = axes[1]
ax2.plot(lengths, conversion_rate, color='steelblue', label='Conversion Rate')
ax2.set_xlabel('Länge der Customer Journey')
ax2.set_ylabel('Wahrscheinlichkeit')
ax2.set_title('Conversion Rate nach Sequenzlänge')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Journey length summary statistics by order
journey_length_comparison = (
    df5
    .groupby("order")["touchpoint_sum"]
    .agg(["count", "mean", "median", "std"])
)

journey_length_comparison

In [ ]:
# Calculate click rate per journey and compare by order status
click_rate = df5["seq_times"].apply(
    lambda times: np.mean([tp["is_click"] for tp in times])
)

click_rate.groupby(df5["order"]).agg(
    ["mean", "median", "std"]
)

In [ ]:
# Top 20 sequence counts and shares (%)
sequence_counts = df5["sequence"].apply(tuple).value_counts()

sequence_summary = pd.DataFrame({
    "count": sequence_counts,
    "share_pct": (sequence_counts / len(df5) * 100).round(2)
})

sequence_summary.head(20)